In [ ]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_enhanced_medical_dataset(n_patients=150, save_path=None):
    """
    Generate enhanced synthetic medical dataset with comprehensive disease categories

    Parameters:
    n_patients (int): Number of patients to generate
    save_path (str): Path where to save the CSV file
    """

    # Generate patient demographics
    patient_ids = [f"P{str(i).zfill(3)}" for i in range(1, n_patients + 1)]
    ages = np.random.randint(18, 85, n_patients)
    genders = np.random.choice(['Male', 'Female'], n_patients)

    # Enhanced disease categories
    disease_types = [
        # General Health
        'Healthy',

        # CBC-related diseases
        'Iron_Deficiency_Anemia', 'Megaloblastic_Anemia', 'Hemolytic_Anemia', 'Aplastic_Anemia',
        'Leukemia', 'Lymphoma', 'Thrombocytopenia', 'Polycythemia',

        # Diabetes and Metabolic diseases
        'Type_1_Diabetes', 'Type_2_Diabetes', 'Gestational_Diabetes', 'Metabolic_Syndrome',
        'Hypoglycemia', 'Diabetic_Nephropathy', 'Diabetic_Neuropathy',

        # Heart and Blood Vessels (Cardiovascular + Lipid + Electrolyte disorders)
        'Hypertension', 'Coronary_Artery_Disease', 'Heart_Failure', 'Myocardial_Infarction',
        'Hyperlipidemia', 'Atherosclerosis', 'Arrhythmia', 'Hyponatremia', 'Hyperkalemia',

        # Nutritional Deficiencies
        'Vitamin_B12_Deficiency', 'Folate_Deficiency', 'Vitamin_D_Deficiency', 'Iron_Deficiency',
        'Protein_Malnutrition', 'Scurvy', 'Rickets',

        # Infections and Immunity
        'Bacterial_Infection', 'Viral_Infection', 'Fungal_Infection', 'Sepsis',
        'Autoimmune_Disease', 'Immunodeficiency', 'Allergic_Reaction'
    ]

    # Assign diseases with realistic prevalence
    disease_probabilities = [
        0.20,  # Healthy
        # CBC-related (20%)
        0.08, 0.03, 0.02, 0.01, 0.015, 0.015, 0.02, 0.015,
        # Diabetes/Metabolic (15%)
        0.02, 0.08, 0.01, 0.02, 0.005, 0.01, 0.005,
        # Cardiovascular (20%)
        0.08, 0.04, 0.02, 0.015, 0.03, 0.02, 0.015, 0.005, 0.005,
        # Nutritional (10%)
        0.02, 0.015, 0.03, 0.015, 0.005, 0.003, 0.002,
        # Infections/Immunity (15%)
        0.04, 0.03, 0.01, 0.005, 0.03, 0.01, 0.015
    ]

    assigned_diseases = np.random.choice(disease_types, n_patients, p=disease_probabilities)

    # Initialize dataset
    medical_data = []

    for i in range(n_patients):
        patient_id = patient_ids[i]
        age = ages[i]
        gender = genders[i]
        disease = assigned_diseases[i]

        # Generate vital signs based on disease
        if disease in ['Hypertension', 'Coronary_Artery_Disease', 'Heart_Failure']:
            systolic_bp = np.random.normal(155, 20)
            diastolic_bp = np.random.normal(95, 15)
        elif disease == 'Hypoglycemia':
            systolic_bp = np.random.normal(100, 15)
            diastolic_bp = np.random.normal(65, 10)
        else:
            systolic_bp = np.random.normal(120, 15)
            diastolic_bp = np.random.normal(80, 10)

        # Heart rate
        if disease in ['Heart_Failure', 'Myocardial_Infarction']:
            heart_rate = np.random.normal(95, 20)
        elif disease == 'Arrhythmia':
            heart_rate = np.random.choice([np.random.normal(45, 8), np.random.normal(120, 15)])
        elif disease in ['Bacterial_Infection', 'Viral_Infection', 'Sepsis']:
            heart_rate = np.random.normal(110, 15)
        else:
            heart_rate = np.random.normal(72, 12)

        # Temperature
        if disease in ['Bacterial_Infection', 'Viral_Infection', 'Sepsis', 'Fungal_Infection']:
            temperature = np.random.normal(102, 2)
        else:
            temperature = np.random.normal(98.6, 0.8)

        # CBC Parameters based on disease
        # Hemoglobin and RBC
        if disease == 'Iron_Deficiency_Anemia':
            hemoglobin = np.random.normal(8.5, 1.5)
            rbc_count = np.random.normal(3.8, 0.5)
            mcv = np.random.normal(72, 8)  # Microcytic
        elif disease == 'Megaloblastic_Anemia':
            hemoglobin = np.random.normal(7.5, 1.2)
            rbc_count = np.random.normal(2.8, 0.4)
            mcv = np.random.normal(115, 10)  # Macrocytic
        elif disease == 'Hemolytic_Anemia':
            hemoglobin = np.random.normal(8.0, 1.0)
            rbc_count = np.random.normal(3.2, 0.4)
            mcv = np.random.normal(85, 8)
        elif disease == 'Aplastic_Anemia':
            hemoglobin = np.random.normal(6.5, 1.0)
            rbc_count = np.random.normal(2.5, 0.3)
            mcv = np.random.normal(88, 8)
        elif disease == 'Polycythemia':
            hemoglobin = np.random.normal(18.5, 2.0)
            rbc_count = np.random.normal(6.5, 0.8)
            mcv = np.random.normal(88, 5)
        elif gender == 'Female':
            hemoglobin = np.random.normal(13.0, 1.2)
            rbc_count = np.random.normal(4.3, 0.4)
            mcv = np.random.normal(87, 5)
        else:
            hemoglobin = np.random.normal(15.0, 1.5)
            rbc_count = np.random.normal(4.8, 0.5)
            mcv = np.random.normal(90, 5)

        # WBC based on condition
        if disease in ['Bacterial_Infection', 'Sepsis']:
            wbc_count = np.random.normal(18000, 5000)
        elif disease in ['Viral_Infection', 'Fungal_Infection']:
            wbc_count = np.random.normal(12000, 3000)
        elif disease == 'Leukemia':
            wbc_count = np.random.choice([np.random.normal(2000, 500), np.random.normal(45000, 15000)])
        elif disease == 'Lymphoma':
            wbc_count = np.random.normal(15000, 8000)
        elif disease in ['Aplastic_Anemia', 'Immunodeficiency']:
            wbc_count = np.random.normal(3000, 1000)
        else:
            wbc_count = np.random.normal(7500, 2000)

        # Platelet count
        if disease == 'Thrombocytopenia':
            platelet_count = np.random.normal(80000, 30000)
        elif disease in ['Leukemia', 'Lymphoma', 'Aplastic_Anemia']:
            platelet_count = np.random.normal(120000, 40000)
        elif disease == 'Polycythemia':
            platelet_count = np.random.normal(600000, 150000)
        else:
            platelet_count = np.random.normal(300000, 80000)

        # Blood chemistry - Glucose and HbA1c
        if disease in ['Type_1_Diabetes', 'Type_2_Diabetes', 'Gestational_Diabetes']:
            glucose = np.random.normal(220, 60)
            hba1c = np.random.normal(9.2, 2.0)
        elif disease == 'Diabetic_Nephropathy':
            glucose = np.random.normal(180, 40)
            hba1c = np.random.normal(8.8, 1.5)
        elif disease == 'Metabolic_Syndrome':
            glucose = np.random.normal(130, 25)
            hba1c = np.random.normal(6.2, 0.8)
        elif disease == 'Hypoglycemia':
            glucose = np.random.normal(55, 15)
            hba1c = np.random.normal(4.8, 0.5)
        else:
            glucose = np.random.normal(95, 15)
            hba1c = np.random.normal(5.4, 0.5)

        # Kidney function
        if disease in ['Diabetic_Nephropathy', 'Heart_Failure']:
            creatinine = np.random.normal(2.8, 1.0)
            bun = np.random.normal(45, 15)
        elif disease == 'Protein_Malnutrition':
            creatinine = np.random.normal(0.7, 0.2)
            bun = np.random.normal(8, 3)
        else:
            creatinine = np.random.normal(1.0, 0.2)
            bun = np.random.normal(15, 5)

        # Liver function
        if disease in ['Viral_Infection', 'Sepsis']:
            alt = np.random.normal(85, 35)
            ast = np.random.normal(90, 40)
            bilirubin = np.random.normal(2.2, 1.0)
        elif disease == 'Hemolytic_Anemia':
            alt = np.random.normal(35, 15)
            ast = np.random.normal(38, 15)
            bilirubin = np.random.normal(4.5, 1.5)  # Indirect bilirubin high
        else:
            alt = np.random.normal(25, 10)
            ast = np.random.normal(28, 12)
            bilirubin = np.random.normal(1.0, 0.3)

        # Lipid profile
        if disease in ['Coronary_Artery_Disease', 'Atherosclerosis', 'Hyperlipidemia']:
            total_cholesterol = np.random.normal(280, 50)
            ldl_cholesterol = np.random.normal(180, 40)
            hdl_cholesterol = np.random.normal(32, 8)
            triglycerides = np.random.normal(250, 70)
        elif disease in ['Type_2_Diabetes', 'Metabolic_Syndrome']:
            total_cholesterol = np.random.normal(240, 40)
            ldl_cholesterol = np.random.normal(160, 30)
            hdl_cholesterol = np.random.normal(35, 8)
            triglycerides = np.random.normal(200, 50)
        else:
            total_cholesterol = np.random.normal(180, 30)
            ldl_cholesterol = np.random.normal(110, 25)
            hdl_cholesterol = np.random.normal(55, 15)
            triglycerides = np.random.normal(120, 40)

        # Electrolytes
        if disease == 'Hyponatremia':
            sodium = np.random.normal(125, 8)
            potassium = np.random.normal(4.0, 0.5)
            chloride = np.random.normal(95, 5)
        elif disease == 'Hyperkalemia':
            sodium = np.random.normal(140, 5)
            potassium = np.random.normal(5.8, 0.8)
            chloride = np.random.normal(102, 5)
        elif disease in ['Heart_Failure', 'Diabetic_Nephropathy']:
            sodium = np.random.normal(135, 8)
            potassium = np.random.normal(5.2, 0.8)
            chloride = np.random.normal(100, 6)
        else:
            sodium = np.random.normal(140, 5)
            potassium = np.random.normal(4.2, 0.5)
            chloride = np.random.normal(102, 4)

        # Inflammatory markers
        if disease in ['Bacterial_Infection', 'Sepsis', 'Autoimmune_Disease']:
            crp = np.random.normal(25, 15)
            esr = np.random.normal(65, 25)
        elif disease in ['Viral_Infection', 'Fungal_Infection']:
            crp = np.random.normal(8, 5)
            esr = np.random.normal(35, 15)
        else:
            crp = np.random.normal(2, 1)
            esr = np.random.normal(12, 8)

        # Vitamins and nutritional markers
        if disease == 'Vitamin_B12_Deficiency':
            vitamin_b12 = np.random.normal(150, 50)
            folate = np.random.normal(8, 3)
            vitamin_d = np.random.normal(30, 10)
        elif disease == 'Folate_Deficiency':
            vitamin_b12 = np.random.normal(400, 100)
            folate = np.random.normal(2.5, 1)
            vitamin_d = np.random.normal(28, 8)
        elif disease == 'Vitamin_D_Deficiency':
            vitamin_b12 = np.random.normal(450, 120)
            folate = np.random.normal(9, 3)
            vitamin_d = np.random.normal(15, 8)
        else:
            vitamin_b12 = np.random.normal(500, 150)
            folate = np.random.normal(12, 4)
            vitamin_d = np.random.normal(35, 12)

        # Iron studies
        if disease in ['Iron_Deficiency_Anemia', 'Iron_Deficiency']:
            serum_iron = np.random.normal(45, 15)
            ferritin = np.random.normal(8, 5)
            transferrin = np.random.normal(380, 50)
        else:
            serum_iron = np.random.normal(120, 30)
            ferritin = np.random.normal(150, 80)
            transferrin = np.random.normal(280, 40)

        # Protein markers
        if disease == 'Protein_Malnutrition':
            total_protein = np.random.normal(5.8, 1.0)
            albumin = np.random.normal(2.8, 0.8)
        else:
            total_protein = np.random.normal(7.2, 0.8)
            albumin = np.random.normal(4.2, 0.6)

        # Immunoglobulins
        if disease == 'Immunodeficiency':
            igg = np.random.normal(400, 150)
            iga = np.random.normal(80, 30)
            igm = np.random.normal(45, 20)
        elif disease == 'Autoimmune_Disease':
            igg = np.random.normal(1800, 400)
            iga = np.random.normal(280, 80)
            igm = np.random.normal(180, 60)
        else:
            igg = np.random.normal(1200, 300)
            iga = np.random.normal(200, 60)
            igm = np.random.normal(120, 40)

        # Urinalysis
        if disease in ['Type_1_Diabetes', 'Type_2_Diabetes', 'Gestational_Diabetes']:
            urine_glucose = np.random.choice(['Positive', 'Negative'], p=[0.8, 0.2])
            urine_protein = np.random.choice(['Trace', 'Negative', '1+'], p=[0.5, 0.3, 0.2])
        elif disease == 'Diabetic_Nephropathy':
            urine_glucose = np.random.choice(['Positive', 'Negative'], p=[0.6, 0.4])
            urine_protein = np.random.choice(['2+', '3+', '1+'], p=[0.5, 0.3, 0.2])
        elif disease in ['Bacterial_Infection', 'Viral_Infection']:
            urine_glucose = 'Negative'
            urine_protein = np.random.choice(['Trace', 'Negative'], p=[0.3, 0.7])
        else:
            urine_glucose = 'Negative'
            urine_protein = 'Negative'

        # BMI calculation
        height = np.random.normal(170 if gender == 'Male' else 162, 10)  # cm
        if disease in ['Type_2_Diabetes', 'Metabolic_Syndrome', 'Heart_Failure']:
            weight = np.random.normal(90, 18)  # kg (higher for these conditions)
        elif disease in ['Protein_Malnutrition', 'Scurvy', 'Rickets']:
            weight = np.random.normal(55, 12)  # kg (lower for malnutrition)
        else:
            weight = np.random.normal(70, 15)  # kg

        bmi = weight / ((height/100) ** 2)

        # Ensure values are within realistic ranges
        def clip_value(value, min_val, max_val):
            return max(min_val, min(max_val, value))

        # Create patient record
        patient_record = {
            'Patient_ID': patient_id,
            'Age': int(age),
            'Gender': gender,
            'Height_cm': round(clip_value(height, 140, 210), 1),
            'Weight_kg': round(clip_value(weight, 35, 180), 1),
            'BMI': round(clip_value(bmi, 12, 50), 1),
            'Primary_Disease': disease,

            # Vital Signs
            'Systolic_BP': round(clip_value(systolic_bp, 80, 220), 1),
            'Diastolic_BP': round(clip_value(diastolic_bp, 45, 130), 1),
            'Heart_Rate': round(clip_value(heart_rate, 40, 150)),
            'Temperature_F': round(clip_value(temperature, 95, 108), 1),
            'Respiratory_Rate': round(clip_value(np.random.normal(16, 4), 10, 35)),
            'O2_Saturation': round(clip_value(np.random.normal(98, 3), 80, 100), 1),

            # Complete Blood Count
            'Hemoglobin': round(clip_value(hemoglobin, 4, 22), 1),
            'RBC_Count': round(clip_value(rbc_count, 2.0, 7.5), 2),
            'WBC_Count': round(clip_value(wbc_count, 1000, 50000)),
            'Platelet_Count': round(clip_value(platelet_count, 50000, 800000)),
            'Hematocrit': round(clip_value(hemoglobin * 3, 15, 65), 1),
            'MCV': round(clip_value(mcv, 65, 125), 1),
            'MCH': round(clip_value(hemoglobin * 2, 18, 40), 1),
            'MCHC': round(clip_value(32 + np.random.normal(0, 2), 28, 38), 1),

            # Blood Chemistry
            'Glucose_mg_dL': round(clip_value(glucose, 40, 450)),
            'HbA1c_percent': round(clip_value(hba1c, 4.0, 16), 1),
            'Creatinine_mg_dL': round(clip_value(creatinine, 0.4, 10), 2),
            'BUN_mg_dL': round(clip_value(bun, 3, 100)),

            # Liver Function
            'ALT_U_L': round(clip_value(alt, 8, 300)),
            'AST_U_L': round(clip_value(ast, 8, 350)),
            'Total_Bilirubin_mg_dL': round(clip_value(bilirubin, 0.1, 15), 2),

            # Lipid Profile
            'Total_Cholesterol_mg_dL': round(clip_value(total_cholesterol, 100, 400)),
            'LDL_mg_dL': round(clip_value(ldl_cholesterol, 40, 300)),
            'HDL_mg_dL': round(clip_value(hdl_cholesterol, 15, 90)),
            'Triglycerides_mg_dL': round(clip_value(triglycerides, 40, 500)),

            # Electrolytes
            'Sodium_mEq_L': round(clip_value(sodium, 120, 155)),
            'Potassium_mEq_L': round(clip_value(potassium, 2.5, 7.0), 1),
            'Chloride_mEq_L': round(clip_value(chloride, 90, 115)),

            # Inflammatory Markers
            'CRP_mg_L': round(clip_value(crp, 0.3, 80), 1),
            'ESR_mm_hr': round(clip_value(esr, 1, 120)),

            # Vitamins and Nutritional Markers
            'Vitamin_B12_pg_mL': round(clip_value(vitamin_b12, 100, 1000)),
            'Folate_ng_mL': round(clip_value(folate, 1, 25), 1),
            'Vitamin_D_ng_mL': round(clip_value(vitamin_d, 5, 80), 1),

            # Iron Studies
            'Serum_Iron_ug_dL': round(clip_value(serum_iron, 20, 200)),
            'Ferritin_ng_mL': round(clip_value(ferritin, 2, 500)),
            'Transferrin_mg_dL': round(clip_value(transferrin, 200, 450)),

            # Protein Markers
            'Total_Protein_g_dL': round(clip_value(total_protein, 4.0, 9.0), 1),
            'Albumin_g_dL': round(clip_value(albumin, 2.0, 5.5), 1),

            # Immunoglobulins
            'IgG_mg_dL': round(clip_value(igg, 200, 2500)),
            'IgA_mg_dL': round(clip_value(iga, 40, 400)),
            'IgM_mg_dL': round(clip_value(igm, 25, 300)),

            # Urinalysis
            'Urine_Glucose': urine_glucose,
            'Urine_Protein': urine_protein,
            'Urine_RBC': np.random.choice(['Negative', 'Few', 'Many'], p=[0.75, 0.2, 0.05]),
            'Urine_WBC': np.random.choice(['Negative', 'Few', 'Many'], p=[0.65, 0.3, 0.05]),
        }

        medical_data.append(patient_record)

    # Create DataFrame
    df = pd.DataFrame(medical_data)

    # Set default save path if not provided
    if save_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"enhanced_medical_dataset_150_patients_{timestamp}.csv"

    # Create directory if it doesn't exist
    save_dir = os.path.dirname(save_path)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Save to CSV
    df.to_csv(save_path, index=False)

    print(f"Enhanced Dataset Shape: {df.shape}")
    print(f"\nDisease Distribution:")
    disease_counts = df['Primary_Disease'].value_counts()
    for disease, count in disease_counts.items():
        print(f"{disease}: {count} ({count/len(df)*100:.1f}%)")

    print(f"\nDataset saved successfully to: {save_path}")
    print(f"File size: {os.path.getsize(save_path)} bytes")

    # Display sample data
    print(f"\nSample of first 3 patients:")
    print(df[['Patient_ID', 'Age', 'Gender', 'Primary_Disease', 'Hemoglobin', 'Glucose_mg_dL', 'Total_Cholesterol_mg_dL']].head(3))

    return df, save_path

# Example usage:
if __name__ == "__main__":
    # Generate enhanced dataset with 150 patients
    df, file_path = generate_enhanced_medical_dataset()

    # For custom path:
    # df, file_path = generate_enhanced_medical_dataset(save_path="./enhanced_medical_dataset.csv")

ValueError: probabilities do not sum to 1